# Instant-1.7B — Experiment 1: LFM3-1.7B with Thinking ON vs OFF

**Question:** how much accuracy does `LiquidAI/LFM3-1.7B` lose when its thinking step is turned off, and how many tokens (and seconds) does that save?

Nobody has published these numbers yet. This notebook measures them on four standard benchmarks:

| Benchmark | Skill | Score |
|---|---|---|
| HumanEval+ | Python coding | passes all (extra-hard) tests |
| MBPP+ | Python coding | passes all (extra-hard) tests |
| GSM8K | grade-school math | final number correct |
| IFEval | following instructions | all instructions followed (strict) |

Same model, same questions, same settings — the **only** difference is `enable_thinking=True` vs `False`.

**Kaggle setup (right sidebar → Session options):** Accelerator **GPU T4 x2**, Internet **On**. If you ran an older version of this notebook in the same session, use **Run → Factory reset** first.

Rough time: install + load ≈ 10 min, scorer self-test ≈ 3 min, experiment ≈ 40–70 min (30 problems per benchmark).

## 1. Install
Only light packages. We use plain `transformers` (like a normal fine-tuning notebook) — no vLLM, so Kaggle's PyTorch is never replaced.
Do **not** import torch before this cell.

In [ ]:
!pip install -q -U transformers accelerate
!pip install -q evalplus absl-py langdetect immutabledict nltk tabulate

## 2. Settings
Change these if you want. `N_PER_BENCH = 100` gives stronger numbers but takes ~3x longer.

In [ ]:
import os
os.environ["USE_TF"] = "0"           # we never use TensorFlow; avoids protobuf import errors on Kaggle

MODEL = "LiquidAI/LFM3-1.7B"
BENCHMARKS = ["humaneval", "mbpp", "gsm8k", "ifeval"]
N_PER_BENCH = 30          # problems per benchmark (seeded random subset, same for both modes)
MAX_NEW_TOKENS = 4096     # same cap for ON and OFF (fair). Answers that hit it count as "truncated"
BATCH_SIZE = 16           # lower to 8 if you see "CUDA out of memory"
SEED = 0

# Liquid's recommended sampling for LFM3 (from the model card)
SAMPLING = {
    "general": dict(temperature=0.1, top_k=50, top_p=1.0, repetition_penalty=1.1),   # code + instructions
    "math":    dict(temperature=1.0, top_k=20, top_p=0.95, repetition_penalty=1.0),  # GSM8K
}

WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.getcwd()
DATA_DIR = os.path.join(WORK, "data")
RESULTS_DIR = os.path.join(WORK, "results")
os.makedirs(DATA_DIR, exist_ok=True); os.makedirs(RESULTS_DIR, exist_ok=True)
print("Working dir:", WORK)

## 3. GPU check & load the model
T4 GPUs have no bfloat16, so we load in float16.

In [ ]:
import time, torch
from transformers import AutoModelForCausalLM, AutoTokenizer

assert torch.cuda.is_available(), "No GPU: set Accelerator to 'GPU T4 x2' in Session options."
for i in range(torch.cuda.device_count()):
    print(f"GPU {i}: {torch.cuda.get_device_name(i)}")

# T4 reports bf16 'supported' through slow emulation, so decide by hardware generation instead
DTYPE = torch.bfloat16 if torch.cuda.get_device_capability(0)[0] >= 8 else torch.float16

tokenizer = AutoTokenizer.from_pretrained(MODEL, padding_side="left")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(MODEL, dtype=DTYPE, device_map={"": 0})
model.eval()

# Tokens that end the assistant's turn
_eos = model.generation_config.eos_token_id
EOS_IDS = set(_eos if isinstance(_eos, list) else [_eos])
if "<|im_end|>" in tokenizer.get_vocab():
    EOS_IDS.add(tokenizer.convert_tokens_to_ids("<|im_end|>"))
EOS_IDS.discard(None)

print(f"Loaded {MODEL} | {sum(p.numel() for p in model.parameters())/1e9:.2f}B params | {DTYPE} | "
      f"GPU memory used: {torch.cuda.memory_allocated(0)/1e9:.1f} GB | end tokens: {EOS_IDS}")

## 4. Template check — does the thinking switch really change the prompt?
With thinking OFF, LFM3's chat template should insert an empty `<think></think>`. If both prompts were identical, the whole experiment would be meaningless, so we stop here in that case.

In [ ]:
def render(messages, thinking):
    # The model's own chat template; enable_thinking is the ON/OFF switch
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True,
                                         enable_thinking=thinking)

def count_tokens(text):
    return len(tokenizer.encode(text, add_special_tokens=False)) if text else 0

_msgs = [{"role": "user", "content": "What is 2+2?"}]
print("===== Thinking ON  =====\n" + render(_msgs, True))
print("===== Thinking OFF =====\n" + render(_msgs, False))
assert render(_msgs, True) != render(_msgs, False), "enable_thinking has no effect — stop and investigate"
print("OK: the two modes produce different prompts.")
with open(os.path.join(RESULTS_DIR, "template_check.txt"), "w") as f:
    f.write("ON:\n" + render(_msgs, True) + "\n\nOFF:\n" + render(_msgs, False))

## 5. Generation function
Generates answers in batches (similar-length prompts together, to save time) and records tokens, time and whether the answer was cut off at the token cap.

In [ ]:
_DROP = [t for t in tokenizer.all_special_tokens if t not in ("<think>", "</think>")]

def _clean(text):
    for t in _DROP:              # remove end-of-turn / padding markers, keep <think> tags
        text = text.replace(t, "")
    return text

@torch.no_grad()
def generate(prompts, sampling, max_new_tokens=MAX_NEW_TOKENS, batch_size=BATCH_SIZE, seed=SEED, label=""):
    torch.manual_seed(seed)
    order = sorted(range(len(prompts)), key=lambda i: len(prompts[i]))
    results, n_batches = [None] * len(prompts), -(-len(prompts) // batch_size)
    do_sample = sampling["temperature"] > 0
    for b, start in enumerate(range(0, len(order), batch_size)):
        idx = order[start:start + batch_size]
        enc = tokenizer([prompts[i] for i in idx], return_tensors="pt", padding=True,
                        add_special_tokens=False).to(model.device)
        t0 = time.perf_counter()
        out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=do_sample,
                             temperature=sampling["temperature"] if do_sample else None,
                             top_k=sampling["top_k"] if do_sample else None,
                             top_p=sampling["top_p"] if do_sample else None,
                             repetition_penalty=sampling["repetition_penalty"],
                             pad_token_id=tokenizer.pad_token_id, eos_token_id=list(EOS_IDS))
        secs = time.perf_counter() - t0
        for row, i in zip(out[:, enc["input_ids"].shape[1]:].tolist(), idx):
            stop = next((k for k, t in enumerate(row) if t in EOS_IDS), None)
            toks = row if stop is None else row[:stop]
            results[i] = {"text": _clean(tokenizer.decode(toks, skip_special_tokens=False)),
                          "n_tokens": len(toks), "truncated": stop is None,
                          "seconds": secs / len(idx)}
        print(f"  {label} batch {b+1}/{n_batches}: {secs:.0f}s, longest answer "
              f"{max(results[i]['n_tokens'] for i in idx)} tokens", flush=True)
    return results

**Batching sanity check.** Padding short prompts in a batch must not change the answer. We generate one question greedily alone and inside a batch and compare. Tiny differences can come from float16 rounding; big ones mean you should set `BATCH_SIZE = 1` (slower but safe).

In [ ]:
_greedy = dict(temperature=0.0, top_k=None, top_p=None, repetition_penalty=1.0)
_q = render([{"role": "user", "content": "Name three primary colors."}], False)
_long = render([{"role": "user", "content": "Explain in detail how a hash map works, " * 20}], False)
_alone = generate([_q], _greedy, max_new_tokens=40, batch_size=1, label="alone")[0]["text"]
_batched = generate([_q, _long], _greedy, max_new_tokens=40, batch_size=2, label="batched")[0]["text"]
print("alone  :", repr(_alone)); print("batched:", repr(_batched))
_same = _alone.split()[:15] == _batched.split()[:15]
print("OK: batching does not change answers." if _same else
      "WARNING: answers differ inside a batch. Compare by eye; if they are clearly different, set BATCH_SIZE = 1.")

## 6. Helpers: separate the thinking from the answer, pull out code and numbers
We only grade the **final answer**. If thinking never finishes (hits the token cap), there is no answer, so it counts as wrong — we never grade half-finished reasoning.

In [ ]:
import re

def split_thinking(raw, prompt_opened_think=False):
    # returns (reasoning, answer, status) with status: closed / unclosed / none
    if "</think>" in raw:
        pre, post = raw.rsplit("</think>", 1)
        return pre.replace("<think>", "").strip(), post.strip(), "closed"
    if "<think>" in raw or prompt_opened_think:
        return raw.split("<think>", 1)[-1].strip(), "", "unclosed"
    return "", raw.strip(), "none"

def prompt_opens_thinking(prompt):
    # True if the chat template already wrote <think> and the model must close it
    tail = prompt.rstrip(); i = tail.rfind("<think>")
    return i != -1 and tail.find("</think>", i) == -1

def leaked_thinking(raw):
    # thinking OFF but the model wrote thinking tags anyway
    return "<think>" in raw or "</think>" in raw

_FENCE = re.compile(r"```[ \t]*([A-Za-z0-9_+-]*)[^\n]*\n(.*?)```", re.DOTALL)

def extract_code(answer, entry_point=None):
    blocks = [(lang.lower(), body) for lang, body in _FENCE.findall(answer)]
    if not blocks:  # no closed fence: take what follows an opening fence, else the whole answer
        m = re.search(r"```[ \t]*[A-Za-z0-9_+-]*[^\n]*\n(.*)", answer, re.DOTALL)
        return (m.group(1) if m else answer).strip("\n")
    py = [b for lang, b in blocks if lang in ("", "python", "py", "python3")] or [b for _, b in blocks]
    if entry_point:
        defining = [b for b in py if re.search(rf"\bdef\s+{re.escape(entry_point)}\s*\(", b)]
        if defining:
            return defining[-1].strip("\n")
    return max(py, key=len).strip("\n")

def leading_imports(code):
    return "\n".join(l for l in code.splitlines() if re.match(r"^(import|from)\s+\S+", l))

_NUM = re.compile(r"-?\d[\d,]*(?:\.\d+)?")

def _boxed(text):
    out, i = [], 0
    while (i := text.find("\\boxed{", i)) != -1:
        j = start = i + len("\\boxed{"); depth = 1
        while j < len(text) and depth:
            depth += {"{": 1, "}": -1}.get(text[j], 0); j += 1
        out.append(text[start:j - 1] if depth == 0 else text[start:]); i = j
    return out

def extract_final_number(text):
    # \boxed{} first, then '####', then 'answer is', then the last number
    boxed = _boxed(text)
    if boxed: cand = boxed[-1]
    elif "####" in text: cand = text.rsplit("####", 1)[-1]
    else:
        m = re.findall(r"answer\s*(?:is|:)\s*([^\n]+)", text, flags=re.IGNORECASE)
        cand = m[-1] if m else text
    nums = _NUM.findall(cand) or _NUM.findall(text)
    try: return float(nums[-1].replace(",", "")) if nums else None
    except ValueError: return None

# quick self-checks of the helpers
assert split_thinking("<think>plan</think>\n\nAnswer: 4") == ("plan", "Answer: 4", "closed")
assert split_thinking("never closed", True)[1:] == ("", "unclosed")
assert prompt_opens_thinking("<|im_start|>assistant\n<think>\n")
assert not prompt_opens_thinking("<|im_start|>assistant\n<think>\n\n</think>\n\n")
assert extract_code("x\n```python\nx=1\n```\n```python\ndef f(a):\n    return a\n```", "f").startswith("def f")
assert extract_final_number("so \\boxed{1,234}") == 1234 and extract_final_number("#### 72") == 72
assert extract_final_number("The answer is 3.5 apples") == 3.5 and extract_final_number("none") is None
print("helpers OK")

# display panel used by the side-by-side views below
import html
from IPython.display import HTML, display

def _panel(title, color, g):
    think = (f"<details><summary>show thinking ({count_tokens(g['reasoning'])} tokens)</summary>"
             f"<pre style='white-space:pre-wrap;color:#555'>{html.escape(g['reasoning'])}</pre></details>") if g["reasoning"] else ""
    return (f"<div style='flex:1;background:#fff;color:#000;padding:12px;border-radius:10px;border-top:5px solid {color}'>"
            f"<b style='color:{color}'>{title}</b> — {g['n_tokens']} tokens, {g['seconds']:.1f}s"
            f"{' — <b>CUT OFF</b>' if g['truncated'] else ''}{think}"
            f"<pre style='white-space:pre-wrap'>{html.escape(g['answer'] or '(no answer)')}</pre></div>")

## 7. Load the benchmarks
Downloaded from their official sources. Each benchmark uses a **seeded random subset**, identical for both modes.

In [ ]:
import gzip, json, random, shutil, urllib.request

FILES = {
    "HumanEvalPlus.jsonl.gz": "https://github.com/evalplus/humanevalplus_release/releases/download/v0.1.10/HumanEvalPlus.jsonl.gz",
    "MbppPlus.jsonl.gz": "https://github.com/evalplus/mbppplus_release/releases/download/v0.2.0/MbppPlus.jsonl.gz",
    "gsm8k_test.jsonl": "https://raw.githubusercontent.com/openai/grade-school-math/master/grade_school_math/data/test.jsonl",
    "ifeval_input_data.jsonl": "https://raw.githubusercontent.com/google-research/google-research/master/instruction_following_eval/data/input_data.jsonl",
}

def download(url, dest, tries=4):
    for k in range(tries):
        try:
            req = url
            if k == tries - 1 and "/releases/download/" in url:   # last try: same file via the GitHub API
                owner, repo, _, _, tag, name = url.split("github.com/")[1].split("/")
                api = json.load(urllib.request.urlopen(f"https://api.github.com/repos/{owner}/{repo}/releases/tags/{tag}", timeout=60))
                asset = next(a["url"] for a in api["assets"] if a["name"] == name)
                req = urllib.request.Request(asset, headers={"Accept": "application/octet-stream"})
            with urllib.request.urlopen(req, timeout=120) as r, open(dest + ".part", "wb") as f:
                shutil.copyfileobj(r, f)
            os.replace(dest + ".part", dest); return
        except Exception as e:
            print(f"  download problem ({e}), retrying..."); time.sleep(3 * (k + 1))
    raise RuntimeError(f"could not download {url}")

for name, url in FILES.items():
    path = os.path.join(DATA_DIR, name)
    if not os.path.exists(path):
        print("downloading", name); download(url, path)

def read_jsonl(name):
    path = os.path.join(DATA_DIR, name)
    with (gzip.open(path, "rt", encoding="utf-8") if path.endswith(".gz") else open(path, encoding="utf-8")) as f:
        return [json.loads(l) for l in f if l.strip()]

def subset(rows, n, seed=SEED):
    if not n or n >= len(rows): return rows
    keep = set(random.Random(seed).sample(range(len(rows)), n))
    return [r for i, r in enumerate(rows) if i in keep]

CODE_INSTRUCTION = ("Return the complete function implementation, including the signature and any "
                    "imports it needs, in a single ```python code block.")
HE_ROWS, MBPP_ROWS = read_jsonl("HumanEvalPlus.jsonl.gz"), read_jsonl("MbppPlus.jsonl.gz")

def load_items(bench, n=N_PER_BENCH):
    items = []
    if bench == "humaneval":
        for r in subset(HE_ROWS, n):
            q = f"Complete the following Python function.\n\n```python\n{r['prompt'].rstrip()}\n```\n\n{CODE_INSTRUCTION}"
            items.append(dict(bench=bench, id=r["task_id"], question=q, entry_point=r["entry_point"], prompt=r["prompt"]))
    elif bench == "mbpp":
        for r in subset(MBPP_ROWS, n):
            task = r["prompt"].strip().strip('"').strip()
            q = f"{task}\n\nThe function must be named `{r['entry_point']}` and pass the test above.\n\n{CODE_INSTRUCTION}"
            items.append(dict(bench=bench, id=r["task_id"], question=q, entry_point=r["entry_point"]))
    elif bench == "gsm8k":
        rows = [dict(bench=bench, id=f"gsm8k/{i}", gold=float(r["answer"].rsplit("####", 1)[-1].strip().replace(",", "")),
                     question=f"{r['question']}\n\nSolve the problem and put the final numeric answer inside \\boxed{{}}.")
                for i, r in enumerate(read_jsonl("gsm8k_test.jsonl"))]
        items = subset(rows, n)
    elif bench == "ifeval":
        items = subset([dict(bench=bench, id=f"ifeval/{r['key']}", question=r["prompt"], row=r)
                        for r in read_jsonl("ifeval_input_data.jsonl")], n)
    return items

ITEMS = {b: load_items(b) for b in BENCHMARKS}
SAMPLING_OF = {"humaneval": "general", "mbpp": "general", "gsm8k": "math", "ifeval": "general"}
for b, its in ITEMS.items():
    print(f"{b:10s} {len(its):4d} problems | example: {its[0]['question'][:90]!r}")

## 8. Scoring
- **Code:** the official `evalplus` checker runs each answer against the original tests plus many extra hard tests (`plus_pass`).
- **Math:** final number must match the reference.
- **Instructions:** Google's original IFEval checker.

In [ ]:
import subprocess, sys, nltk
for p in ("punkt", "punkt_tab"):
    nltk.download(p, quiet=True)

# Google's IFEval checker code (downloaded once)
IFEVAL_DIR = os.path.join(WORK, "vendor", "instruction_following_eval")
os.makedirs(IFEVAL_DIR, exist_ok=True)
for f in ["instructions.py", "instructions_registry.py", "instructions_util.py", "evaluation_lib.py"]:
    if not os.path.exists(os.path.join(IFEVAL_DIR, f)):
        download(f"https://raw.githubusercontent.com/google-research/google-research/master/instruction_following_eval/{f}",
                 os.path.join(IFEVAL_DIR, f))
open(os.path.join(IFEVAL_DIR, "__init__.py"), "a").close()
sys.path.insert(0, os.path.dirname(IFEVAL_DIR))
from instruction_following_eval import evaluation_lib as ifeval_lib

def code_solution(it, answer):
    code = extract_code(answer, it["entry_point"])
    if it["bench"] == "humaneval":
        if f"def {it['entry_point']}" not in code:      # model wrote only the body
            return it["prompt"] + code
        return leading_imports(it["prompt"]) + "\n\n" + code
    return code

def score_code(bench, items, answers, tag):
    # evalplus needs every problem in the file; the ones we did not pick get an empty solution and are ignored
    rows, fname, env_key = ((HE_ROWS, "HumanEvalPlus.jsonl.gz", "HUMANEVAL_OVERRIDE_PATH") if bench == "humaneval"
                            else (MBPP_ROWS, "MbppPlus.jsonl.gz", "MBPP_OVERRIDE_PATH"))
    work = os.path.join(RESULTS_DIR, "scoring", tag); os.makedirs(work, exist_ok=True)
    samples = os.path.join(work, f"{bench}_samples.jsonl")
    sol = {it["id"]: code_solution(it, a) for it, a in zip(items, answers)}
    with open(samples, "w") as f:
        for r in rows:
            f.write(json.dumps({"task_id": r["task_id"], "solution": sol.get(r["task_id"], "")}) + "\n")
    res_file = samples.replace(".jsonl", "_eval_results.json")
    if os.path.exists(res_file): os.remove(res_file)      # evalplus would silently reuse old results
    env = {**os.environ, env_key: os.path.join(DATA_DIR, fname)}
    # Generous time limits (same for both modes): on a busy machine evalplus's defaults can time out
    # even the official reference solution on its huge extra tests.
    p = subprocess.run([sys.executable, "-m", "evalplus.evaluate", bench, "--samples", samples,
                        "--parallel", str(max(1, (os.cpu_count() or 2) // 2)),
                        "--min-time-limit", "4", "--gt-time-limit-factor", "8"],
                       env=env, capture_output=True, text=True)
    if p.returncode != 0 or not os.path.exists(res_file):
        raise RuntimeError("evalplus failed:\n" + p.stdout[-2000:] + p.stderr[-2000:])
    ev = json.load(open(res_file))["eval"]
    out = []
    for it in items:
        r = ev[it["id"]][0]; base = r["base_status"] == "pass"
        out.append({"correct": base and r["plus_status"] == "pass", "base_pass": base})
    return out

def score(bench, items, answers, tag="run"):
    if bench in ("humaneval", "mbpp"):
        return score_code(bench, items, answers, tag)
    if bench == "gsm8k":
        out = []
        for it, a in zip(items, answers):
            pred = extract_final_number(a)
            out.append({"correct": pred is not None and abs(pred - it["gold"]) <= 1e-4 * max(1, abs(it["gold"])), "pred": pred})
        return out
    out = []
    for it, a in zip(items, answers):
        r = it["row"]
        inp = ifeval_lib.InputExample(key=r["key"], instruction_id_list=r["instruction_id_list"], prompt=r["prompt"], kwargs=r["kwargs"])
        strict = ifeval_lib.test_instruction_following_strict(inp, {r["prompt"]: a})
        loose = ifeval_lib.test_instruction_following_loose(inp, {r["prompt"]: a})
        out.append({"correct": strict.follow_all_instructions, "loose": loose.follow_all_instructions})
    return out
print("scorers ready")

## 9. Scorer self-test (no model needed)
Before spending GPU time we prove the scoring is right: **reference solutions must score ~100%**, **wrong answers 0%**. For IFEval we replay GPT-4's published responses, which should score about **77%** (the paper reports 76.9%).

In [ ]:
_gpt4_path = os.path.join(DATA_DIR, "ifeval_gpt4.jsonl")
if not os.path.exists(_gpt4_path):
    download("https://raw.githubusercontent.com/google-research/google-research/master/instruction_following_eval/data/input_response_data_gpt4_20231107_145030.jsonl", _gpt4_path)
GPT4 = {r["prompt"]: r["response"] for r in read_jsonl("ifeval_gpt4.jsonl")}
HE_BY_ID, MBPP_BY_ID = {r["task_id"]: r for r in HE_ROWS}, {r["task_id"]: r for r in MBPP_ROWS}

def reference_answer(it, good=True):
    if it["bench"] == "humaneval":
        r = HE_BY_ID[it["id"]]
        return (f"```python\n{r['prompt']}{r['canonical_solution']}```" if good
                else f"```python\ndef {r['entry_point']}(*a, **k):\n    return None\n```")
    if it["bench"] == "mbpp":
        return f"```python\n{MBPP_BY_ID[it['id']]['canonical_solution']}\n```" if good else "```python\nprint('hi')\n```"
    if it["bench"] == "gsm8k":
        g = it["gold"]; return f"\\boxed{{{int(g) if g == int(g) else g}}}" if good else "\\boxed{-999999}"
    return GPT4.get(it["row"]["prompt"], "") if good else ""

for b in BENCHMARKS:
    its = ITEMS[b]
    good = sum(s["correct"] for s in score(b, its, [reference_answer(i, True) for i in its], "selftest_good")) / len(its)
    bad = sum(s["correct"] for s in score(b, its, [reference_answer(i, False) for i in its], "selftest_bad")) / len(its)
    print(f"{b:10s} reference answers: {good:6.1%}   wrong answers: {bad:5.1%}")
    assert bad == 0, f"{b}: wrong answers should score 0"
    assert good >= (0.5 if b == "ifeval" else 0.95), f"{b}: reference answers scored too low"
print("\nScorer self-test passed.")

## 10. First look: one question, both modes
Same question, thinking ON vs OFF, side by side (click "show thinking" to see the reasoning).

In [ ]:
def ask(question, thinking, sampling="general", max_new_tokens=MAX_NEW_TOKENS):
    p = render([{"role": "user", "content": question}], thinking)
    g = generate([p], SAMPLING[sampling], max_new_tokens=max_new_tokens, batch_size=1, label="ON" if thinking else "OFF")[0]
    reasoning, answer, status = split_thinking(g["text"], prompt_opens_thinking(p))
    return dict(reasoning=reasoning, answer=answer, status=status, **g)

def compare(question, sampling="general"):
    on, off = ask(question, True, sampling), ask(question, False, sampling)
    display(HTML(f"<div style='background:#000;padding:16px;border-radius:12px;font-family:sans-serif'>"
                 f"<div style='background:#e0e0e0;color:#000;padding:10px;border-radius:8px;margin-bottom:12px'>"
                 f"<b>QUESTION:</b> {html.escape(question)}</div><div style='display:flex;gap:12px'>"
                 f"{_panel('THINKING ON', '#e67e22', on)}{_panel('THINKING OFF', '#007bff', off)}</div></div>"))
    return on, off

_ = compare("A shop sells pens at 3 for $4. How much do 18 pens cost? Put the final answer in \\boxed{}.", "math")

## 11. Run the experiment
Every benchmark, both modes. Answers are saved to `results/` as they finish, so if Kaggle disconnects you can re-run this cell and it continues where it stopped.

In [ ]:
MODES = {"ON": True, "OFF": False}
RECORDS = []    # one row per (mode, benchmark, question)

for mode, thinking in MODES.items():
    for b in BENCHMARKS:
        its = ITEMS[b]
        cache = os.path.join(RESULTS_DIR, f"{mode}_{b}.jsonl")
        prompts = [render([{"role": "user", "content": it["question"]}], thinking) for it in its]
        cached = [json.loads(l) for l in open(cache)] if os.path.exists(cache) else None
        if cached and [c["id"] for c in cached] == [it["id"] for it in its]:
            print(f"== {b} [thinking {mode}]: using saved answers"); gens = cached
        else:
            print(f"== {b} [thinking {mode}]: generating {len(its)} answers", flush=True)
            gens = generate(prompts, SAMPLING[SAMPLING_OF[b]], label=f"{b}/{mode}")
            for it, g in zip(its, gens): g["id"] = it["id"]
            with open(cache, "w") as f:
                f.writelines(json.dumps(g) + "\n" for g in gens)
        rows = []
        for it, p, g in zip(its, prompts, gens):
            reasoning, answer, status = split_thinking(g["text"], prompt_opens_thinking(p))
            rows.append(dict(mode=mode, bench=b, id=it["id"], question=it["question"], raw=g["text"],
                             reasoning=reasoning, answer=answer, think_status=status,
                             n_tokens=g["n_tokens"], reasoning_tokens=count_tokens(reasoning),
                             truncated=g["truncated"], seconds=g["seconds"],
                             leak=(not thinking) and leaked_thinking(g["text"])))
        for r, s in zip(rows, score(b, its, [r["answer"] for r in rows], tag=f"{mode}_{b}")):
            r.update(correct=bool(s["correct"]))
        RECORDS += rows
        print(f"   {b} [thinking {mode}] accuracy = {sum(r['correct'] for r in rows)/len(rows):.1%}, "
              f"mean tokens = {sum(r['n_tokens'] for r in rows)/len(rows):.0f}")

with open(os.path.join(RESULTS_DIR, "all_answers.jsonl"), "w") as f:
    f.writelines(json.dumps(r) + "\n" for r in RECORDS)
print("\nDone:", len(RECORDS), "answers scored.")

## 12. Results

In [ ]:
import pandas as pd
df = pd.DataFrame(RECORDS)

summary = (df.groupby(["bench", "mode"], sort=False)
             .agg(n=("correct", "size"), accuracy=("correct", "mean"),
                  mean_tokens=("n_tokens", "mean"), median_tokens=("n_tokens", "median"),
                  reasoning_tokens=("reasoning_tokens", "mean"), truncated=("truncated", "mean"),
                  leaked_thinking=("leak", "mean"), sec_per_answer=("seconds", "mean"))
             .reset_index())
display(summary.style.format({"accuracy": "{:.1%}", "truncated": "{:.1%}", "leaked_thinking": "{:.1%}",
                              "mean_tokens": "{:.0f}", "median_tokens": "{:.0f}", "reasoning_tokens": "{:.0f}",
                              "sec_per_answer": "{:.1f}"}))

on = summary[summary["mode"] == "ON"].set_index("bench"); off = summary[summary["mode"] == "OFF"].set_index("bench")
gap = pd.DataFrame({"accuracy ON": on["accuracy"], "accuracy OFF": off["accuracy"],
                    "gap (points)": (on["accuracy"] - off["accuracy"]) * 100,
                    "OFF keeps % of ON": off["accuracy"] / on["accuracy"].replace(0, float("nan")) * 100,
                    "tokens ON": on["mean_tokens"], "tokens OFF": off["mean_tokens"],
                    "OFF uses x fewer tokens": on["mean_tokens"] / off["mean_tokens"].replace(0, float("nan"))})
gap.loc["AVERAGE"] = gap.mean(numeric_only=True)
print("\nThinking ON vs OFF")
display(gap.style.format({"accuracy ON": "{:.1%}", "accuracy OFF": "{:.1%}", "gap (points)": "{:+.1f}",
                          "OFF keeps % of ON": "{:.0f}%", "tokens ON": "{:.0f}", "tokens OFF": "{:.0f}",
                          "OFF uses x fewer tokens": "{:.1f}x"}))

summary.to_csv(os.path.join(RESULTS_DIR, "summary.csv"), index=False)
gap.to_csv(os.path.join(RESULTS_DIR, "on_vs_off.csv"))
with open(os.path.join(RESULTS_DIR, "summary.md"), "w") as f:
    f.write(f"# {MODEL}: thinking ON vs OFF (n={N_PER_BENCH}/benchmark, max_new_tokens={MAX_NEW_TOKENS}, seed={SEED})\n\n")
    f.write(summary.to_markdown(index=False, floatfmt=".3f") + "\n\n" + gap.to_markdown(floatfmt=".2f") + "\n")

## 13. Charts

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

x = np.arange(len(BENCHMARKS)); w = 0.38
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
for k, (col, title, fmt) in enumerate([("accuracy", "Accuracy (higher is better)", "{:.0%}"),
                                        ("mean_tokens", "Mean tokens per answer (lower is cheaper)", "{:.0f}")]):
    for j, (mode, color) in enumerate([("ON", "#e67e22"), ("OFF", "#007bff")]):
        vals = [summary[(summary.bench == b) & (summary["mode"] == mode)][col].iloc[0] for b in BENCHMARKS]
        bars = ax[k].bar(x + (j - 0.5) * w, vals, w, label=f"thinking {mode}", color=color)
        ax[k].bar_label(bars, labels=[fmt.format(v) for v in vals], fontsize=8)
    ax[k].set_xticks(x, BENCHMARKS); ax[k].set_title(title); ax[k].legend()
    ax[k].spines[["top", "right"]].set_visible(False)
ax[0].set_ylim(0, 1.1)
plt.suptitle(f"{MODEL}: thinking ON vs OFF"); plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "on_vs_off.png"), dpi=150); plt.show()

## 14. Where does thinking OFF fail?
For each question: did both modes get it right, only ON, only OFF, or neither? The **"only ON right"** questions are exactly what fine-tuning must fix — they show what the model loses without thinking.

In [ ]:
wide = df.pivot_table(index=["bench", "id"], columns="mode", values="correct", aggfunc="first").reset_index()
_on, _off = wide["ON"].astype(bool), wide["OFF"].astype(bool)
wide["outcome"] = np.select([_on & _off, _on & ~_off, ~_on & _off],
                            ["both right", "only ON right", "only OFF right"], "both wrong")
display(pd.crosstab(wide["bench"], wide["outcome"], margins=True))

lost = wide[wide.outcome == "only ON right"]
print(f"\n{len(lost)} questions solved with thinking but failed without it. Two examples:")
by_key = {(r["mode"], r["id"]): r for r in RECORDS}
for _, row in lost.head(2).iterrows():
    a_on, a_off = by_key[("ON", row["id"])], by_key[("OFF", row["id"])]
    display(HTML(f"<div style='background:#000;padding:14px;border-radius:12px;font-family:sans-serif;margin:8px 0'>"
                 f"<div style='background:#e0e0e0;color:#000;padding:8px;border-radius:8px;margin-bottom:10px'>"
                 f"<b>{row['bench']} {row['id']}</b><pre style='white-space:pre-wrap'>{html.escape(a_on['question'][:1200])}</pre></div>"
                 f"<div style='display:flex;gap:12px'>{_panel('ON (correct)', '#2e7d32', a_on)}{_panel('OFF (wrong)', '#c62828', a_off)}</div></div>"))
wide.to_csv(os.path.join(RESULTS_DIR, "per_question_outcomes.csv"), index=False)

## 15. Speed test: how long does a user wait?
One question at a time (batch size 1), like a real chat on a laptop or phone. This is the number users feel.

In [ ]:
speed = []
for b in BENCHMARKS:
    it = ITEMS[b][0]
    for mode, thinking in MODES.items():
        g = ask(it["question"], thinking, SAMPLING_OF[b])
        speed.append(dict(bench=b, mode=mode, tokens=g["n_tokens"], seconds=g["seconds"],
                          tokens_per_sec=g["n_tokens"] / max(g["seconds"], 1e-9)))
speed = pd.DataFrame(speed)
display(speed.pivot(index="bench", columns="mode", values=["seconds", "tokens", "tokens_per_sec"]).round(1))
t_on, t_off = speed[speed["mode"] == "ON"].seconds.sum(), speed[speed["mode"] == "OFF"].seconds.sum()
print(f"Waiting time for these {len(BENCHMARKS)} questions: ON {t_on:.0f}s vs OFF {t_off:.0f}s  "
      f"({t_on / max(t_off, 1e-9):.1f}x faster without thinking)")
speed.to_csv(os.path.join(RESULTS_DIR, "speed_test.csv"), index=False)

## 16. Interactive sandbox
Ask anything and flip the thinking switch.

In [ ]:
import ipywidgets as widgets

box = widgets.Text(placeholder="Ask LFM3 something...", description="Message:", layout=widgets.Layout(width="70%"))
think = widgets.Checkbox(value=False, description="Thinking ON")
both = widgets.Checkbox(value=False, description="Compare both")
out = widgets.Output()

def _on_submit(_):
    q = box.value.strip()
    if not q: return
    with out:
        if both.value:
            compare(q)
        else:
            g = ask(q, think.value)
            display(HTML(f"<div style='display:flex'>{_panel('THINKING ' + ('ON' if think.value else 'OFF'), '#e67e22' if think.value else '#007bff', g)}</div>"))
    box.value = ""

box.on_submit(_on_submit)
display(widgets.HBox([box, think, both]), out)

## 17. Save results
Everything is in `results/` and zipped to `results.zip` (Output panel → download). Send `summary.md`, `on_vs_off.png` and `per_question_outcomes.csv` for review.

In [ ]:
shutil.make_archive(os.path.join(WORK, "results"), "zip", RESULTS_DIR)
print("saved", os.path.join(WORK, "results.zip"))
print(open(os.path.join(RESULTS_DIR, "summary.md")).read())